In [80]:
import torch
from torch import nn

from tqdm import tqdm_notebook

In [81]:
mu = 0
L = 10
A = torch.Tensor([[mu, 0], [0, L]])
x0 = torch.Tensor([10, 10])

A

tensor([[ 0.,  0.],
        [ 0., 10.]])

In [82]:
class Noise(nn.Module):
    def __init__(self, iterations):
        super().__init__()
        self.noise = nn.Linear(2, iterations)

    def add_i(self, i):
        return self.noise.weight[i]


In [87]:
iterations = 100
noise = Noise(iterations)

device = 'cpu'

x = x0.to(device)
AA = A.to(device)
alpha = 0.9
coef = ((1 - alpha) / (1 + alpha)) ** (3 / 2)
h = 1 / L * coef
h = 4 / ((L ** 0.5 + mu ** 0.5) ** 2)
beta = ((L ** 0.5 - mu ** 0.5) / (L ** 0.5 + mu ** 0.5)) ** 2
learning_rate = 1e-2
optimizer = torch.optim.Adam(noise.parameters(), lr=learning_rate)

iter_learn = 100

losses = []


for _ in tqdm_notebook(range(iter_learn)):
    x = x0.to(device)
    AA = A.to(device)
    vals = []
    for j in range(iterations):
        n = noise.add_i(j)
        gr = AA @ x
        g = gr + n / torch.norm(n) * torch.norm(gr) * alpha
        x = x - h * g
        v = x @ AA @ x
        vals.append(v.item())
        # print(vals)
    l = -x @ AA @ x
    losses.append(l.item())
    l.backward()
    optimizer.step()



/tmp/ipykernel_49113/2712476743.py:21: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for _ in tqdm_notebook(range(iter_learn)):


  0%|          | 0/100 [00:00<?, ?it/s]

In [88]:
losses

[nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan]

In [83]:
mu = 1
L = 100
A = torch.Tensor([[mu, 0], [0, L]])
x0 = torch.Tensor([10, 10])


In [108]:
iterations = 1000
noise = Noise(iterations)

device = 'cpu'

x = x0.to(device)
AA = A.to(device)
alpha = 2 * (mu / L) ** 0.5
alpha = 0.001
print(alpha)
coef = ((1 - alpha) / (1 + alpha)) ** (3 / 2)
h = 1 / L * coef
h = 4 / ((L ** 0.5 + mu ** 0.5) ** 2)
beta = ((L ** 0.5 - mu ** 0.5) / (L ** 0.5 + mu ** 0.5)) ** 2
print(beta)
beta = 0.000001
learning_rate = 1e-2
optimizer = torch.optim.Adam(noise.parameters(), lr=learning_rate)

iter_learn = 1

losses = []


def method(x):
    vals = []
    p = torch.zeros(x.shape[0]).to(device)
    for j in range(iterations):
        n = noise.add_i(j)
        gr = AA @ x
        g = gr + n / torch.norm(n) * torch.norm(gr) * alpha
        x, p = x - h * g - beta * (x - p), x
        v = x @ AA @ x
        vals.append(v.item())
    return x



for _ in tqdm_notebook(range(iter_learn)):
    x = x0.to(device)
    AA = A.to(device)
    x = method(x)
    l = -x @ AA @ x
    losses.append(l.item())
    l.backward()
    optimizer.step()


0.001
0.6694214876033059


/tmp/ipykernel_60241/3520161837.py:39: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for _ in tqdm_notebook(range(iter_learn)):


  0%|          | 0/1 [00:00<?, ?it/s]

In [109]:
losses

[nan]

In [50]:
class StateNoise(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.grad_mat = nn.Linear(dim, dim)
        self.state_mat = nn.Linear(dim, dim)
        self.prev_state_mat = nn.Linear(dim, dim)
        self.comb = nn.Linear(3, 1, bias=False)

    def forward(self, g, x, x_prev):
        # print(g, self.grad_mat)
        g_ = self.grad_mat(g)
        x_ = self.state_mat(x)
        x_prev_ = self.prev_state_mat(x_prev)
        st = torch.stack([g_, x_, x_prev_]).T
        return (st @ self.comb.weight.T).T[0]
        

class Noise(nn.Module):
    def __init__(self, iterations):
        super().__init__()
        self.noise = nn.Linear(2, iterations)

    def add_i(self, i):
        return self.noise.weight[i]


In [57]:
mu = 0.1
L = 100
A = torch.Tensor([[mu, 0], [0, L]])
x0 = torch.Tensor([100, 100])

A

tensor([[  0.1000,   0.0000],
        [  0.0000, 100.0000]])

In [78]:
iterations = 100

noise = Noise(iterations)

device = 'cpu'

x = x0.to(device)
B = A.to(device)
alpha = (mu / L) ** 0.5
alpha = 0.
#alpha = 2
print(alpha)
coef = ((1 - alpha) / (1 + alpha)) ** (3 / 2)
h = 1 / L * coef
h = 4 / ((L ** 0.5 + mu ** 0.5) ** 2)
beta = ((L ** 0.5 - mu ** 0.5) / (L ** 0.5 + mu ** 0.5)) ** 2
learning_rate = 1e-1
optimizer = torch.optim.Adam(noise.parameters(), lr=learning_rate)

iter_learn = 5000
iter_learn = 5


losses = []


def method(x: torch.Tensor):
    x = x.to(device)
    u = x.to(device)
    y = x.to(device)
    aa = 10000
    aa = 1
    Aseq = 0
    for j in range(iterations):
        aseq_new = (j + 2) / 2 / L / aa
        Aseq_new = Aseq + aseq_new
        x = (Aseq * y + aseq_new * u) / Aseq_new
        gr = B @ x
        #u1 = u - aseq_new * gr
        #y1 = (Aseq * y + aseq_new * u1) / Aseq_new
        # n = noise.forward(gr, x, y1)
        n = noise.add_i(j)
        # print(n)
        g = gr + n / torch.norm(n) * torch.norm(gr) * alpha
        u = u - aseq_new * g
        y = (Aseq * y + aseq_new * u) / Aseq_new
    return y


for _ in tqdm_notebook(range(iter_learn)):
    x = x0.to(device)
    AA = A.to(device)
    x = method(x)
    l = -x @ AA @ x
    losses.append(l.item())
    l.backward()
    # print(noise.noise.weight.grad)
    optimizer.step()


0.0


/tmp/ipykernel_60241/3563806037.py:50: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for _ in tqdm_notebook(range(iter_learn)):


  0%|          | 0/5 [00:00<?, ?it/s]

In [79]:
losses

[-5.303343772888184,
 -5.303343772888184,
 -5.303343772888184,
 -5.303343772888184,
 -5.303343772888184]